# 03 - Value Implementation

这本 `course` 不是压缩版提纲。课堂 notebook 会先把直觉、例子、代码和图跑通，再进入最后的 Predict / Modify 检查。

学习顺序：先读这一页的主线，遇到代码就运行；最后再做底部的小检查。真正写作业时进入同目录的 `homework.ipynb`。

In [ ]:
from pathlib import Path
import sys, math, random, inspect


def find_repo_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / 'micrograd' / 'engine.py').exists():
            return q
    return p

ROOT = find_repo_root()
PROJECT_ROOT = ROOT
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from micrograd.engine import Value
from micrograd.nn import Neuron, Layer, MLP

try:
    import matplotlib.pyplot as plt
    MATPLOTLIB_AVAILABLE = True
except ModuleNotFoundError:
    plt = None
    MATPLOTLIB_AVAILABLE = False

try:
    import torch
except ModuleNotFoundError:
    torch = None


def near(a, b, eps=1e-6):
    return abs(a - b) < eps


def assert_close(actual, expected, name='value', eps=1e-9):
    assert abs(actual - expected) < eps, f'{name}: expected {expected}, got {actual}'


def todo_guard(values, message='请先填写 TODO，再运行检查。'):
    if any(v is None for v in values):
        print(message)
        return False
    return True

from course.checks import qa_check


def show_svg(path):
    path = Path(path)
    try:
        from IPython.display import SVG, display
        display(SVG(filename=str(path)))
    except Exception:
        print('图已生成，但当前环境不能内嵌显示。请打开:', path.resolve())

print('project root:', ROOT)


## 练习方式：先读源码，再补小块

这一节不是让你照着源码抄一遍，而是按这个顺序练：

```text
1. 先观察真实源码做了什么
2. 用小数字把 self / other / out 代进去
3. 在留空练习里补 1-3 行关键代码
4. 运行 qa_check 看是否理解对
5. 卡住时先看提示，最后才看答案
```

重点不是背代码，而是搞清楚：每个运算在前向时保存了什么，反向时怎么把 `out.grad` 传给父节点。

## 本节路线图：四次接触同一个思想

同一个概念会出现四次，难度一点点加：

| 次数 | 你做什么 | 目标 |
|---|---|---|
| 1 | 看 micrograd 真源码 | 知道真实实现长什么样 |
| 2 | 把 `a+b` 代入 `self/other/out` | 消除 Python 语法障碍 |
| 3 | 做留空练习并运行 `qa_check` | 确认自己能补关键规则 |
| 4 | 写一个小版本并测试 | 从“看懂”变成“能写” |

这才是这一节的学习方式：源码给直觉，练习给掌握。

## 1. Value Is A Number Plus Autograd Metadata

先看构造函数：


In [ ]:
print(inspect.getsource(Value.__init__))


`Value` 里最重要的字段：

```text
data       当前节点的数值
       grad       最终输出对当前节点的导数/偏导，初始是 0
_backward  一个函数，负责把当前节点的 grad 传给父节点
_prev      父节点集合，也就是当前节点从哪些节点算出来
_op        当前节点由什么运算产生，比如 +、*、ReLU
```

叶子节点最简单：


In [ ]:
a = Value(2.0)
print('a.data =', a.data)
print('a.grad =', a.grad)
print('a._prev =', a._prev)
print('a._op =', repr(a._op))
print('a._backward =', a._backward)


## 2. Addition Creates A New Value Node

看 `__add__`：


In [ ]:
print(inspect.getsource(Value.__add__))


##### 这段代码做了两件事：

```text
1. 前向：out.data = self.data + other.data
2. 反向规则：如果 out 收到梯度 out.grad，就把它原样传给 self 和 other
```

因为：

$$
out = x + y
$$

所以：

$$
\frac{\partial out}{\partial x}=1, \quad \frac{\partial out}{\partial y}=1
$$

也就是：

```python
self.grad += out.grad
other.grad += out.grad
```

做个小实验：


In [ ]:
a = Value(2.0)
b = Value(3.0)
c = a + b

print('c =', c)
print('c._op =', c._op)
print('c._prev =', c._prev)

print('parents data =', [v.data for v in c._prev])

c.backward()
print('a.grad =', a.grad)
print('b.grad =', b.grad)


## 2.1 用具体例子理解 `__add__`

如果你对 Python 不太熟，先不用管“魔法方法”这个名字。只记住：

```python
c = a + b
```

当 `a` 是 `Value` 时，Python 会自动调用：

```python
a.__add__(b)
```

所以在源码里：

```python
def __add__(self, other):
```

对应关系就是：

```text
self  = a
other = b
```

如果：

```python
a = Value(2.0)
b = Value(3.0)
c = a + b
```

那么可以把 `__add__` 临时翻译成下面这个更直白的版本：

```python
def add(a, b):
    b = b if isinstance(b, Value) else Value(b)

    c = Value(a.data + b.data, (a, b), '+')

    def _backward():
        a.grad += c.grad
        b.grad += c.grad

    c._backward = _backward
    return c
```

这段做了三件事：

```text
1. 算前向结果：c.data = a.data + b.data = 5
2. 记录来源：c._prev = {a, b}, c._op = '+'
3. 绑定反向规则：以后 c 收到 grad，就把它传回 a 和 b
```


### Python 补给站：`__add__` 为什么会被调用

Python 看到：

```python
c = a + b
```

如果 `a` 是 `Value`，它会尝试调用：

```python
a.__add__(b)
```

所以读源码时永远可以这样代入：

```text
self  = 加号左边的 Value，也就是 a
other = 加号右边的值，也就是 b
out   = 这次加法新创建出来的结果节点 c
```

这个代入法也适用于乘法：

```text
c = a * b  等价于  a.__mul__(b)
```

In [ ]:
a = Value(2.0)
b = Value(3.0)
c = a + b

print('a =', a)
print('b =', b)
print('c = a + b =', c)
print()
print('c.data =', c.data)
print('c.grad =', c.grad)
print('c._op =', c._op)
print('c._prev data =', [parent.data for parent in c._prev])
print('c._backward =', c._backward)


## 2.2 手动调用加法节点的 `_backward`

正常情况下，我们调用：

```python
c.backward()
```

它会自动设置：

```python
c.grad = 1
```

然后再调用：

```python
c._backward()
```

为了看清楚，我们这里手动做一次：

```text
c = a + b
加法局部导数：dc/da = 1, dc/db = 1
所以：a.grad += c.grad, b.grad += c.grad
```


In [ ]:
a = Value(2.0)
b = Value(3.0)
c = a + b

print('before manual backward')
print('a.grad =', a.grad)
print('b.grad =', b.grad)
print('c.grad =', c.grad)

# Pretend c is the final output. The final output's gradient to itself is 1.
c.grad = 1
c._backward()

print('\nafter c.grad = 1 and c._backward()')
print('a.grad =', a.grad)
print('b.grad =', b.grad)
print('c.grad =', c.grad)


## 2.3 为什么要处理 `a + 3`

源码第一行：

```python
other = other if isinstance(other, Value) else Value(other)
```

是为了让下面这种写法也能工作：

```python
a = Value(2.0)
c = a + 3
```

因为普通数字 `3` 没有 `.data`、`.grad`、`._backward` 这些属性，所以要先包装成：

```python
Value(3)
```

这样加法逻辑就统一了：

```text
Value + Value
```


In [ ]:
a = Value(2.0)
c = a + 3

print('c =', c)
print('c.data =', c.data)
print('parents data =', [parent.data for parent in c._prev])

c.backward()
print('a.grad =', a.grad)


## 3. Multiplication Creates A New Value Node

看 `__mul__`：


In [ ]:
print(inspect.getsource(Value.__mul__))


乘法的局部导数是：

$$
out = xy
$$

$$
\frac{\partial out}{\partial x}=y, \quad \frac{\partial out}{\partial y}=x
$$

所以 micrograd 写成：

```python
self.grad += other.data * out.grad
other.grad += self.data * out.grad
```

这里的 `out.grad` 是链式法则里“从后面传来的梯度”。

小实验：


In [ ]:
a = Value(2.0)
b = Value(3.0)
c = a * b

c.backward()
print('c.data =', c.data)
print('a.grad =', a.grad)  # dc/da = b = 3
print('b.grad =', b.grad)  # dc/db = a = 2


## 4. Why `_backward` Is A Closure

`_backward` 是定义在 `__add__` / `__mul__` 里面的函数。

它能记住当时参与运算的变量：

```python
self
other
out
```

这叫闭包。直观理解：

```text
前向计算时，把“将来怎么反传”这张小纸条贴到 out 节点上。
等 backward 真正发生时，再读这张纸条，把梯度传回父节点。
```

下面手动调用 `_backward`，看它确实会修改父节点的 grad：


In [ ]:
a = Value(2.0)
b = Value(3.0)
c = a * b

# Normally backward() sets the final output grad to 1.
# Here we do it manually to inspect the local backward rule.
c.grad = 1
c._backward()

print('after c._backward():')
print('a.grad =', a.grad)
print('b.grad =', b.grad)


## 5. Why Use `+=` Instead Of `=`

因为一个变量可能通过多条路径影响最终输出。

例如：

$$
L = a + a
$$

数学上：

$$
\frac{dL}{da}=2
$$

如果反向传播时不用 `+=` 累加，就会丢掉其中一条路径的贡献。


In [ ]:
a = Value(2.0)
L = a + a
L.backward()

print('L.data =', L.data)
print('a.grad =', a.grad)


## 6. Backward Needs Topological Order

现在看 `backward()` 源码：


In [ ]:
print(inspect.getsource(Value.backward))


为什么要拓扑排序？

因为反向传播时，必须保证：

```text
先处理最终输出
再处理它的父节点
再处理父节点的父节点
```

如果顺序乱了，某个节点可能还没收到完整梯度就开始往前传。

`topo` 是从叶子到输出的顺序，`reversed(topo)` 才是反向传播执行顺序。


In [ ]:
def build_demo_graph():
    a = Value(2.0)
    b = Value(3.0)
    c = a * b
    d = c + a
    L = d * 2
    return {'a': a, 'b': b, 'c': c, 'd': d, 'L': L}

nodes = build_demo_graph()
L = nodes['L']

# Rebuild the topological order, just like Value.backward does.
topo = []
visited = set()

def build_topo(v):
    if v not in visited:
        visited.add(v)
        for child in v._prev:
            build_topo(child)
        topo.append(v)

build_topo(L)

def name_of(value):
    for name, node in nodes.items():
        if node is value:
            return name
    return f'Value({value.data})'

print('topo:', ' -> '.join(name_of(v) for v in topo))
print('backward order:', ' -> '.join(name_of(v) for v in reversed(topo)))


## 6.1 `_prev` 是 set，所以 topo 顺序不是唯一的

在 `Value.__init__` 里有这一行：

```python
self._prev = set(_children)
```

`set` 的特点是：

```text
1. 会去重
2. 不保证遍历顺序
```

所以对于：

```python
c = a * b
```

逻辑上：

```text
c._prev = {a, b}
```

但是执行：

```python
for child in c._prev:
```

可能先看到 `a`，也可能先看到 `b`。这不是按变量名字母序，也不是按 `data` 大小排序。

因此 `topo` 的输出不是唯一答案。重点不是：

```text
topo 必须等于 a -> b -> c -> d -> Value(2) -> L
```

而是：

```text
父节点必须出现在子节点前面
```

也就是：

```text
a、b 在 c 前面
c、a 在 d 前面
d、Value(2) 在 L 前面
```

只要满足这个依赖关系，就是合法的拓扑顺序。


In [ ]:
nodes = build_demo_graph()
L = nodes['L']

topo = []
visited = set()

def build_topo(v):
    if v not in visited:
        visited.add(v)
        for child in v._prev:
            build_topo(child)
        topo.append(v)

build_topo(L)

# Give the anonymous constant Value(2) a readable name for this check.
def stable_name(value):
    for name, node in nodes.items():
        if node is value:
            return name
    return f'const({value.data})'

names = [stable_name(v) for v in topo]
position = {v: i for i, v in enumerate(topo)}

print('one valid topo order:')
print(' -> '.join(names))
print()
print('dependency checks:')
for child in topo:
    print('-- child', stable_name(child), child._prev)
    for parent in child._prev:
        print(f'{stable_name(parent):>8} before {stable_name(child):>8}:', position[parent] < position[child])


如果你看到 `topo` 顺序变了，不要紧张。

比如下面这些都可能合法：

```text
a -> b -> c -> d -> const(2) -> L
b -> a -> c -> d -> const(2) -> L
const(2) -> a -> b -> c -> d -> L
```

只要父节点在子节点前面，反向传播就能正常工作。

`reversed(topo)` 用来反向执行时，也不要求唯一顺序。对同一层互不依赖的节点，谁先谁后通常不影响最终梯度。


## 7. Trace One Backward Pass

我们用同一个例子：

$$
L = 2(ab+a)
$$

前向：

```text
a=2, b=3, c=6, d=8, L=16
```

反向后应该得到：

```text
a.grad = 8
b.grad = 4
```


In [ ]:
nodes = build_demo_graph()

def show(nodes, title):
    print(title)
    for name in ['a', 'b', 'c', 'd', 'L']:
        v = nodes[name]
        parents = []
        for p in v._prev:
            parents.append(name_of_node(p, nodes))
        print(f'{name:>2} | data={v.data:>5} | grad={v.grad:>5} | op={v._op or "leaf":>4} | parents={parents}')

def name_of_node(value, nodes):
    for name, node in nodes.items():
        if node is value:
            return name
    return f'Value({value.data})'

print(nodes)
show(nodes, 'before backward')
nodes['L'].backward()
print()
show(nodes, 'after backward')


## What To Remember

- 跑通主线。
- 说清楚本节的输入、输出、梯度或训练含义。
- 完成底部课堂检查后再进入 homework。

## 课堂检查：先预测，再改一行

这一段保留 `course` 的隐藏检查。你应该先自己填，再展开提示或答案。

## Predict - Python 会调用谁？

In [ ]:
# 填写说明：
# - 完成 student_add_object_probe，用运行结果证明 self/other/out 的关系。
# - 返回顺序固定：c.data、c._op、a 是否在 c._prev、b 是否在 c._prev。
# - 运行本 cell，看 qa_check 的提示。

def student_add_object_probe():
    a = Value(2.0)
    b = Value(3.0)
    c = a + b
    # TODO: return c_data, c_op, a_is_parent, b_is_parent
    pass


qa_check('qa_check_class_03_predict', globals())


<details><summary>Show / Hide 本题提示</summary>

- `c = a + b` 运行后，`c` 就是源码里创建出来的 `out`。
- `c._prev` 里应该能找到参与加法的两个父节点。

</details>


<details><summary>Show / Hide 本题答案</summary>

```python
def student_add_object_probe():
    a = Value(2.0)
    b = Value(3.0)
    c = a + b
    return c.data, c._op, a in c._prev, b in c._prev
```

</details>


## Run - 看产生的新节点

In [ ]:
from micrograd.engine import Value

a = Value(2.0)
b = Value(3.0)
c = a + b
print('c.data:', c.data)
print('c._op:', c._op)
print('parents:', c._prev)

## 拆开看 - `_backward` 是延后执行的局部规则

加法节点的局部规则：上游梯度原样传给两个输入。乘法节点的局部规则：乘另一个输入的 data。

In [ ]:
print('add backward: self.grad += out.grad; other.grad += out.grad')
print('mul backward: self.grad += other.data*out.grad; other.grad += self.data*out.grad')

## Modify - `a * 3` 里的普通数字

In [ ]:
# 填写说明：
# - 完成 student_mul_wrap_probe，观察 a * 3 时普通数字如何参与计算图。
# - 返回顺序固定：out.data、out._op、len(out._prev)。
# - 运行本 cell，看 qa_check 的提示。

def student_mul_wrap_probe():
    a = Value(2.0)
    out = a * 3
    # TODO: return out_data, out_op, parent_count
    pass


qa_check('qa_check_class_03_modify', globals())


<details><summary>Show / Hide 本题提示</summary>

- `a * 3` 能跑，是因为源码会先把普通数字 `3` 包成一个 `Value(3)`。
- 包完以后，它和 `a * b` 一样，都是两个父节点做乘法。

</details>


<details><summary>Show / Hide 本题答案</summary>

```python
def student_mul_wrap_probe():
    a = Value(2.0)
    out = a * 3
    return out.data, out._op, len(out._prev)
```

</details>
